# 🎯 [K09_M05] 자료에 없는 질문 처리하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K09/k09_m05_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·상담 (Level 2) / K09 부대 정보 검색 시스템 |
| **미션** | 미션 05. 자료에 없는 질문 처리하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 프롬프트 엔지니어링 및 팩트체크 검증 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k09_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K09", "../uploads/K09"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k09_m01~m10) 준비 완료!")



---

## 🎯 미션 05. 자료에 없는 질문 처리하기
**소요시간**: 약 1.5시간

![미션 5 자료 밖 질문 차단 및 안전 Fallback 규칙 다이어그램](/uploads/common/K09_m05_safety_fallback.png)

### 📋 미션 개요 및 배경
안내 자료에 명시되지 않은 질문이 입력되었을 때 AI가 허위 답변을 생성(할루시네이션)하는 현상을 방지하는 예외 처리 규칙을 수립합니다. '월급일', '휴무 행사' 등 자료 밖 질문을 입력하여 AI의 오작동 패턴을 확인하고, 정보 부재 시 "자료에 없음" 명시와 함께 행정반·당직실 등 공식 부서로 연계하도록 지시문을 고도화합니다. 시스템의 안전성과 신뢰성을 확보하는 거절 프롬프팅 기법을 습득합니다.

### 🎯 미션 학습목표
'월급날', '행사 일정' 등 자료 밖 질문을 입력하여 AI의 오작동 패턴을 관찰하고, 정보 부재 시 임의 추론을 차단하며 "자료에 없음" 명시 및 담당 부서를 안내하도록 규칙을 추가합니다. **이를 통해 시스템의 안전 경계를 설정하고 오안내 위험을 선제 차단하는 안전 프롬프팅 및 Fallback(예외 대응) 처리 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 질문 카드 17~19를 넣는다.
#### 이 단계의 역할
안내 자료에 전혀 수록되지 않은 질문 카드 17~19번(월급일, 식단표, 체육대회 일정 등)을 입력하여 AI의 허위 답변 생성을 포착합니다.

#### 해야 할 일

<li>실습 데이터에서 '자료 밖 질문'으로 설계된 카드 17, 18, 19번을 확인합니다.

</li><li>현재 프롬프트에 17~19번 질문을 입력하고 AI의 응답을 수집합니다.

</li><li>AI가 자료에 없음에도 당당하게 가상의 날짜나 일정을 창작해 내는 환각 응답을 기록합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[자료 밖 질문 취약점 노출 질의]:** "[프롬프트 본문]: {기존 자료 기반 프롬프트}
[자료 밖 질문 17]: 이번 달 병사 급여 입금일과 시간을 알려줘.
[자료 밖 질문 18]: 다음 주 연대 체육대회 축구 일정과 대진표를 알려줘.
[자료 밖 질문 19]: 부대 내 종교시설 세례식 일정이 어떻게 되나요?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **환각 현상 포착:** 자료에 없는 내용임에도 AI가 임의로 그럴듯한 답변을 지어냈는지 확인했는가?
- **위험성 인지:** 이러한 거짓 답변이 부대 행정 혼선에 미치는 위험성을 기록했는가?

#### 핵심 확인 지표
<blockquote>자료 밖 질문 3건에 대한 AI 환각 응답 기록표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>자료 밖 질문 1차 질의 결과 기록지

</li>

#### 🔹 Step 2. 자료에 없으면 안내 문구로 답하게 조건을 넣는다.
#### 이 단계의 역할
정보 부재 시 임의 추론 생성을 즉각 중단하고 표준 거절 문구와 함께 실무 담당 부서(행정반/당직실)로 연결하는 Fallback 규칙을 설계합니다.

#### 해야 할 일

<li>"자료에 해당 정보가 전혀 없을 경우 '해당 정보는 안내 자료에 없습니다'라고 출력하라"는 원칙을 명문화합니다.

</li><li>"추가 확인이 필요할 경우 [담당 문의처: 본부 행정반 033-XXX-XXXX 또는 당직실]로 문의하라"는 고정 안내문을 작성합니다.

</li><li>프롬프트의 [예외 대응 규칙] 섹션에 해당 Fallback 지침을 정식 삽입합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[예외 대응 Fallback 프롬프트 설계]:** "[예외 처리 규칙 추가]:
- 사용자의 질문이 위 [부대 안내 자료]에 언급되지 않은 내용이라면 절대 임의로 지어내지 마십시오.
- 자료에 없는 질문에는 반드시 다음과 같이 고정된 서식으로 답하십시오:
"⚠️ 안내 자료 미등록 정보: 해당 문의는 부대 생활 안내 자료에 등록되어 있지 않습니다. 정확한 규정 및 일정은 본부 행정반 또는 지휘통제실(당직실)로 문의하시기 바랍니다.""

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **거절 원칙 명확성:** 애매한 추측성 답변 없이 단호하게 정보 부재를 인정하도록 지시되었는가?
- **공식 창구 안내:** 실무 장병이 찾아갈 수 있는 공식 담당 부서(행정반/당직실)가 명시되었는가?

#### 핵심 확인 지표
<blockquote>자료 밖 질문 처리 규칙(Fallback 지침문)

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>자료 밖 질문 처리 규칙(필수 산출물)

</li>

#### 🔹 Step 3. 전후를 비교한다.
#### 이 단계의 역할
예외 처리 규칙 적용 전(환각 답변)과 적용 후(안전 거절 답변)를 1:1로 비교하여 오안내 위험 차단 성과를 입증합니다.

#### 해야 할 일

<li>규칙 적용 후 동일한 17~19번 질문을 재질의하여 새 응답을 받습니다.

</li><li>적용 전의 허위 답변과 적용 후의 표준 거절 답변을 나란히 배치한 전후 비교표를 작성합니다.

</li><li>정상 질문(1~5번)을 다시 넣어 예외 규칙 추가로 인해 기존 정상 답변 기능이 훼손되지 않았는지(회귀 테스트) 확인합니다.

</li>

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[예외 처리 적용 전후 비교 질의]:** "다음 질문 3건에 대해 예외 처리 규칙이 적용된 프롬프트로 재응답하고, 이전 응답과의 안전성 개선 사항을 비교 요약해 줘."

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **환각 차단율 100%:** 3개 질문 모두에서 지어낸 정보가 완벽히 소멸되었는가?
- **정상 기능 유지:** 기존에 잘 답변하던 정상 질문 5건이 여전히 온전하게 출력되는가?

#### 핵심 확인 지표
<blockquote>자료 밖 질문 처리 전후 비교 검증표

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>처리 전후 비교(필수 산출물)

</li>

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할
자료에 있는 정보와 없는 정보가 섞인 복합 질문(혼합 질문)에 대해 분기 처리하는 고급 프롬프팅을 테스트합니다.

#### 해야 할 일
'자료에 없으면 가장 가까운 시설을 안내하라'는 조건을 추가해 본다.

#### 💡 AI 프롬프트 질의 예시
<blockquote>**[혼합 질문 분기 처리 질의]:** "[혼합 질문 처리 규칙]: 한 질문 안에 자료에 있는 내용과 없는 내용이 섞여 있을 경우, 자료에 있는 부분은 정상 답변하고 없는 부분만 골라 'OO 정보는 자료에 없어 행정반 문의가 필요합니다'로 분리 서술할 것.
[질문]: 의무실 진료 시간은 몇 시까지이고, 다음 주 사격 훈련 일정은 언제인가요?"

</blockquote>
#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **부분 분기 정확성:** 알고 있는 정보와 모르는 정보가 뒤섞이지 않고 깔끔히 분리되었는가?

#### 핵심 확인 지표
<blockquote>복합 혼합 질문 분기 처리 시험 메모

</blockquote>
#### 학습자가 제출해야 하는 내용

<li>심화 도전 과제 수행 메모

</li>

### 📝 실습 서식 및 모범 작성 양식 (참고 템플릿)
# [K09_M05] 자료에 없는 질문 처리하기 실습 가이드 및 작성 서식

## 1. 자료 밖 질문 처리 규칙 (Fallback 템플릿)

```markdown
[예외 대응 처리 수칙]:
1. 사용자의 질문 내용이 제공된 [부대 생활 안내 자료]에 명시되어 있지 않은 경우, 절대 임의로 상상하거나 추론하여 지어내지 말라.
2. 자료에 없는 질문에는 반드시 아래의 표준 서식으로 단호하게 답변하라:
   "⚠️ 안내 자료 미등록 정보: 해당 문의 사항은 부대 생활 안내 자료에 등록되어 있지 않습니다. 정확한 규정 및 세부 일정은 본부 행정반(내선 101) 또는 당직사령실로 문의하시기 바랍니다."
```

---

## 2. 예외 처리 규칙 적용 전후 비교 검증표

| 질문 ID | 질문 내용 | 적용 전 AI 답변 (환각 현상) | 적용 후 AI 답변 (안전 Fallback) | 판정 |
| :---: | :--- | :--- | :--- | :---: |
| **Q17** | 병사 월급 입금일 | "매월 10일 정오에 나라사랑카드 계좌로 입금됩니다." (가상 날조) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |
| **Q18** | 연대 체육대회 일정 | "다음 달 15일 연병장에서 축구 예선이 열립니다." (가상 날조) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |
| **Q19** | 개방행사 면회실 예약 | "최소 1주일 전까지 신청서를 내야 합니다." (임의 추론) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |

### 📦 필수 학습자료 및 실습 팁
**미션 수행 필수 학습자료**

- **자료 범위 밖 예외 질문 카드 3종 (카드 17~19):** 월급일, 체육대회 일정, 세례식 등 자료표에 없는 함정 질의 세트
- **안전 Fallback(예외 대응) 처리 규칙 템플릿:** 정보 부재 시 "자료에 없음" 명시 및 행정반/당직실 공식 안내문 서식
- **[학습 가이드] 안전 프롬프팅 및 Fallback(거절) 메커니즘:** AI가 모르는 것을 솔직하게 인정하고 공식 채널로 토스하게 만드는 프롬프트 안전 경계 설계법

**실습 팁:** 거절 지침을 줄 때는 단순히 "답하지 마"라고만 하면 AI가 침묵하거나 일반론을 말하므로, 반드시 "정확히 이 문장으로 답하라"며 표준 거절 문구를 지정해 주어야 합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가 기준:** [필수] 예외 처리·환각 방지 — 자료 밖 질문에 대해 거짓 정보를 지어내지 않고 표준 문구 및 담당 부서를 안내하는 Fallback 규칙을 구현했는가? (PASS / FAIL)

#### 미션 PASS 조건

<li>자료 밖 질문 3건에 대한 취약점 포착 및 응답 기록

</li><li>공식 문의처(행정반/당직실)가 연계된 표준 Fallback 지침 수립

</li><li>예외 규칙 적용 전후 비교표 작성을 통한 환각 차단율 100% 달성

</li>

#### 채점자 판정 메모
<blockquote>자료 밖 질문(월급일, 체육대회 등)을 던졌을 때 단 한 문장의 거짓 정보도 생성하지 않고 완벽히 거절 및 공식 부서로 토스하는지 확인한다.

</blockquote>





In [ ]:
# [미션 05. 자료에 없는 질문 처리하기] 모범 프롬프트 및 완성 결과 기록
mission_05_prompt = """
[자료 밖 질문 처리 규칙(Fallback) 및 안전 방어 지시문 수립]
[역할]: 육군 한빛대대 부대 생활 정보 질의응답 도우미
[예외 대응 처리 수칙]:
1. 사용자의 질문 내용이 제공된 [부대 생활 안내 자료]에 명시되어 있지 않은 경우, 절대 임의로 상상하거나 추론하여 지어내지 말라.
2. 자료에 없는 질문에는 반드시 아래의 표준 서식으로 단호하게 답변하라:
   "⚠️ 안내 자료 미등록 정보: 해당 문의 사항은 부대 생활 안내 자료에 등록되어 있지 않습니다. 정확한 규정 및 확인은 소속 부대 행정반 또는 담당 부서에 직접 확인하시기 바랍니다."

[테스트 질문 세트]:
- Q17: 이번 달 병사 급여 입금일과 시간을 알려줘.
- Q18: 다음 주 연대 체육대회 축구 일정과 대진표를 알려줘.
- Q19: 체단실 벤치프레스 바벨 무게를 추가로 구매할 계획이 있는지 알려줘.
"""

mission_05_result = """
# [K09_M05] 자료에 없는 질문 처리하기 실습 가이드 및 작성 서식

## 1. 자료 밖 질문 처리 규칙 (Fallback 템플릿)

```markdown
[예외 대응 처리 수칙]:
1. 사용자의 질문 내용이 제공된 [부대 생활 안내 자료]에 명시되어 있지 않은 경우, 절대 임의로 상상하거나 추론하여 지어내지 말라.
2. 자료에 없는 질문에는 반드시 아래의 표준 서식으로 단호하게 답변하라:
   "⚠️ 안내 자료 미등록 정보: 해당 문의 사항은 부대 생활 안내 자료에 등록되어 있지 않습니다. 정확한 규정 및 세부 일정은 본부 행정반(내선 101) 또는 당직사령실로 문의하시기 바랍니다."
```

---

## 2. 예외 처리 규칙 적용 전후 비교 검증표

| 질문 ID | 질문 내용 | 적용 전 AI 답변 (환각 현상) | 적용 후 AI 답변 (안전 Fallback) | 판정 |
| :---: | :--- | :--- | :--- | :---: |
| **Q17** | 병사 월급 입금일 | "매월 10일 정오에 나라사랑카드 계좌로 입금됩니다." (가상 날조) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |
| **Q18** | 연대 체육대회 일정 | "다음 달 15일 연병장에서 축구 예선이 열립니다." (가상 날조) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |
| **Q19** | 개방행사 면회실 예약 | "최소 1주일 전까지 신청서를 내야 합니다." (임의 추론) | "⚠️ 안내 자료 미등록 정보: ...본부 행정반 문의" | **PASS** |
"""

# 렌더러 실행 (모범 실습 결과를 시각적으로 확인합니다)
show_result("미션 05. 자료에 없는 질문 처리하기", mission_05_prompt, mission_05_result)

